# 02 — GenerationConfig: All Sampling Strategies

Every knob on `GenerationConfig` explained and compared:

| Strategy | Key fields |
|----------|------------|
| Greedy | `do_sample=False` |
| Temperature | `temperature`, `do_sample=True` |
| Top-K | `top_k`, `apply_top_k_sampling=True` |
| Top-P (nucleus) | `top_p`, `apply_top_p_sampling=True` |
| Repetition penalty | `repetition_penalty`, `apply_repetition_penalty=True` |
| KV cache | `use_kv_cache=True` |
| Log-probs | `return_logprobs=True` |

> **Colab users:** Run the setup cell, restart runtime, then continue.

In [ ]:
import os, sys, subprocess

def _is_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

def _install():
    IN_COLAB = _is_colab()
    if IN_COLAB:
        repo = 'MyLLM'
        if not os.path.exists(repo):
            print('Cloning repository...')
            r = subprocess.run(
                ['git', 'clone', 'https://github.com/silvaxxx1/MyLLM.git', repo],
                capture_output=True, text=True
            )
            if r.returncode != 0:
                print('Clone failed:\n', r.stderr); return
        print('Installing myllm...')
        r = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', repo],
            capture_output=True, text=True
        )
    else:
        root = os.path.abspath(os.path.join(os.getcwd(), '..'))
        print(f'Installing from {root} ...')
        r = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-e', root],
            capture_output=True, text=True
        )
    if r.returncode != 0:
        print('Install FAILED. Error output:')
        print(r.stdout[-3000:] if r.stdout else '')
        print(r.stderr[-3000:] if r.stderr else '')
    else:
        msg = 'Restart runtime now (Runtime → Restart runtime).' if IN_COLAB else 'Ready.'
        print('Done!', msg)

try:
    import myllm
    print(f'myllm {myllm.__version__} already installed.')
except ImportError:
    _install()

In [ ]:
# ── Load model once, reuse across all cells ───────────────────────────────────
import torch
from myllm import LLM, ModelConfig, GenerationConfig
from transformers import GPT2Tokenizer as HFTokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'

llm = LLM(config=ModelConfig.from_name('gpt2-small'), device=device)
llm.load('gpt2-small', 'gpt2')

tokenizer = HFTokenizer.from_pretrained('gpt2')
tokenizer.pad_token_id = tokenizer.eos_token_id

PROMPT = 'Scientists recently discovered that'

def gen(cfg: GenerationConfig) -> str:
    return llm.generate_text(PROMPT, tokenizer, cfg)['text']

print('Ready. device:', device)

## 1. Greedy decoding

Always picks the highest-probability token. Deterministic — same output every run.

In [ ]:
cfg = GenerationConfig(
    max_length=50,
    do_sample=False,
    use_kv_cache=True,
    use_optimized_sampler=False,
)
print('Greedy:')
print(gen(cfg))

## 2. Temperature

- `< 1.0` → sharper → more focused
- `> 1.0` → flatter → more random
- `= 1.0` → unmodified model distribution

In [ ]:
for temp in [0.3, 0.7, 1.0, 1.5]:
    cfg = GenerationConfig(
        max_length=40, temperature=temp, do_sample=True,
        apply_top_k_sampling=False, apply_top_p_sampling=False,
        use_kv_cache=True,
    )
    print(f'temperature={temp}:')
    print(' ', gen(cfg))
    print()

## 3. Top-K sampling

Keep only the top K tokens at each step and renormalise.

In [ ]:
for k in [5, 20, 50, 200]:
    cfg = GenerationConfig(
        max_length=40, temperature=1.0, do_sample=True,
        top_k=k, apply_top_k_sampling=True,
        apply_top_p_sampling=False,
        use_kv_cache=True, use_optimized_sampler=True,
    )
    print(f'top_k={k}:')
    print(' ', gen(cfg))
    print()

## 4. Top-P (nucleus) sampling

Keep the smallest set of tokens whose cumulative probability ≥ p.

In [ ]:
for p in [0.5, 0.8, 0.95, 1.0]:
    cfg = GenerationConfig(
        max_length=40, temperature=1.0, do_sample=True,
        top_p=p, apply_top_k_sampling=False,
        apply_top_p_sampling=True,
        use_kv_cache=True, use_optimized_sampler=True,
    )
    print(f'top_p={p}:')
    print(' ', gen(cfg))
    print()

## 5. Top-K + Top-P combined (recommended default)

In [ ]:
cfg = GenerationConfig(
    max_length=80, temperature=0.8, do_sample=True,
    top_k=50, top_p=0.95,
    apply_top_k_sampling=True, apply_top_p_sampling=True,
    use_kv_cache=True, use_optimized_sampler=True,
)
print('Top-K=50 + Top-P=0.95 + temp=0.8:')
print(gen(cfg))

## 6. Repetition penalty

Divides the logit of any already-seen token by `repetition_penalty`. Values > 1.0 discourage repeating.

In [ ]:
for penalty in [1.0, 1.2, 1.5, 2.0]:
    cfg = GenerationConfig(
        max_length=60, temperature=0.9, do_sample=True,
        top_k=50, apply_top_k_sampling=True,
        repetition_penalty=penalty, apply_repetition_penalty=True,
        use_kv_cache=True, use_optimized_sampler=True,
    )
    print(f'repetition_penalty={penalty}:')
    print(' ', gen(cfg))
    print()

## 7. KV cache speed comparison

In [ ]:
import time, torch

input_ids = tokenizer.encode(PROMPT, return_tensors='pt').to(device)

for use_kv in [True, False]:
    cfg = GenerationConfig(
        max_length=50, temperature=1.0, do_sample=True,
        top_k=50, apply_top_k_sampling=True,
        use_kv_cache=use_kv, use_optimized_sampler=True,
    )
    with torch.no_grad(): llm.generate(input_ids, cfg)  # warmup

    t0 = time.perf_counter()
    with torch.no_grad(): out = llm.generate(input_ids, cfg)
    elapsed = time.perf_counter() - t0
    n = out['tokens'].numel()
    print(f'use_kv_cache={str(use_kv):<5}  {n} tokens  {elapsed:.3f}s  {n/elapsed:.1f} tok/s')

## 8. Log-probabilities

In [ ]:
import torch

cfg = GenerationConfig(
    max_length=15, temperature=0.8, do_sample=True,
    top_k=50, apply_top_k_sampling=True,
    use_kv_cache=True, return_logprobs=True,
    use_optimized_sampler=True,
)

input_ids = tokenizer.encode(PROMPT, return_tensors='pt').to(device)
output    = llm.generate(input_ids, cfg)

prompt_len       = input_ids.shape[1]
generated_tokens = output['tokens'][0][prompt_len:]
logprobs         = output['logprobs'][0]

print(f'{"Token":<20}  Log-prob   Prob')
print('-' * 42)
for tok_id, lp in zip(generated_tokens.tolist(), logprobs.tolist()):
    word = tokenizer.decode([tok_id])
    prob = torch.exp(torch.tensor(lp)).item()
    print(f'{repr(word):<20}  {lp:>8.3f}   {prob:.4f}')